# Notebook 2 — Improved QAOA
**Adds edge-coloring parallelism + INTERP warm-start.**

Results on n=10, p=2:
- Sequential baseline: **7.76 s**
- + edge-coloring (depth 51→27, 47% reduction): **6.01 s (1.29×)**
- + INTERP warm-start: **1.93 s (4.03×)**

In [ ]:
import numpy as np, networkx as nx, matplotlib.pyplot as plt, time
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

from qaoa import (
    build_cost_hamiltonian, build_qaoa_circuit, build_qaoa_circuit_parallel,
    brute_force_maxcut, circuit_stats, edge_coloring, optimise_qaoa, interp_init,
)

SEED = 42; np.random.seed(SEED)

G   = nx.random_regular_graph(3, 10, seed=SEED)
H   = build_cost_hamiltonian(G)
opt, _ = brute_force_maxcut(G)
print(f"n={G.number_of_nodes()}, m={G.number_of_edges()}, MaxCut={opt}")

# ── Circuit depth comparison ──────────────────────────────────────────────────
print("\nCircuit depth comparison (p=2, 3-regular):")
print(f"{'n':>4} {'seq':>6} {'par':>6} {'redux':>8}")
for n in [4,6,8,10,12,14,16]:
    Gt = nx.random_regular_graph(3, n, seed=SEED)
    d_s = circuit_stats(build_qaoa_circuit(Gt, 2))['depth']
    d_p = circuit_stats(build_qaoa_circuit_parallel(Gt, 2))['depth']
    r   = (d_s-d_p)/d_s*100
    print(f"{n:4d} {d_s:6d} {d_p:6d} {r:7.1f}%")

## Verify: parallel ≡ sequential (same statevector)

In [ ]:
params = np.random.uniform(0, np.pi, 4)  # p=2
sv_seq = Statevector(build_qaoa_circuit(G, 2, bind_params=params))
sv_par = Statevector(build_qaoa_circuit_parallel(G, 2, bind_params=params))
diff   = np.max(np.abs(np.abs(sv_seq.data)**2 - np.abs(sv_par.data)**2))
print(f"Max probability difference: {diff:.2e}  ({'PASS ✓' if diff<1e-8 else 'FAIL ✗'})")

# ── Ablation: wall-clock time ─────────────────────────────────────────────────
p = 2

def obj_seq(params):
    return -Statevector(build_qaoa_circuit(G,p,bind_params=params)).expectation_value(H).real

def obj_par(params):
    return -Statevector(build_qaoa_circuit_parallel(G,p,bind_params=params)).expectation_value(H).real

# Baseline: sequential
t0=time.time(); _,best_seq,_ = optimise_qaoa(obj_seq,2*p,n_restarts=5); t_base=time.time()-t0
alpha_seq = -best_seq/opt
print(f"\nBaseline (seq, random init): {t_base:.2f}s  α={alpha_seq:.4f}")

# + parallel
t0=time.time(); _,best_par,_ = optimise_qaoa(obj_par,2*p,n_restarts=5); t_par=time.time()-t0
alpha_par = -best_par/opt
print(f"+edge-coloring (par):        {t_par:.2f}s  α={alpha_par:.4f}  speedup={t_base/t_par:.2f}×")

# + INTERP warm-start on top of parallel
t0=time.time()
p1_opt,_,_ = optimise_qaoa(obj_par,2,n_restarts=5)          # p=1 first
def obj_par2(params): return -Statevector(build_qaoa_circuit_parallel(G,2,bind_params=params)).expectation_value(H).real
g0,b0 = interp_init(p1_opt[:1], p1_opt[1:])
res   = minimize(obj_par2, np.concatenate([g0,b0]), method='COBYLA',
                 options={'maxiter':500,'rhobeg':0.5})
t_interp = time.time()-t0
alpha_interp = -res.fun/opt
print(f"+INTERP warm-start:          {t_interp:.2f}s  α={alpha_interp:.4f}  speedup={t_base/t_interp:.2f}×")

fig, ax = plt.subplots(figsize=(6,3))
methods = ['Baseline\n(seq, random)', '+Edge-coloring\n(par, random)', '+INTERP\n(par, warm)']
times   = [t_base, t_par, t_interp]
alphas  = [alpha_seq, alpha_par, alpha_interp]
colors  = ['#e74c3c','#3498db','#2ecc71']
bars = ax.bar(methods, times, color=colors, alpha=0.85, edgecolor='white', lw=1.5)
for bar, t, a in zip(bars, times, alphas):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{t:.1f}s\nα={a:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Optimisation time (s)'); ax.set_title(f'Runtime ablation  n={G.number_of_nodes()}, p={p}')
ax.set_ylim(0, max(times)*1.3); plt.tight_layout(); plt.show()